<a href="https://colab.research.google.com/github/nguyenanhthu000000/Machine-learning/blob/main/06.%20SVM/NguyenAnhThu_2474802010381_lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

# 24 điểm, hai cụm tách hẳn nhau
X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]],
                  cluster_std=0.9, random_state=0)
y = 1 - y  # Lớp 1 là cụm dưới bên trái, giống hình trên slide

# C rất lớn nghĩa là không cho điểm nào nằm trong lề
svm = SVC(kernel="linear", C=1000)
svm.fit(X, y)

w = svm.coef_[0]
b = svm.intercept_[0]
print(f"w1 = {w[0]:.3f}, w2 = {w[1]:.3f}, b = {b:.3f}")

do_dai_w = np.sqrt(w[0]**2 + w[1]**2)
print(f"Do rong cua le = 2 / {do_dai_w:.4f} = {2/do_dai_w:.4f}")
print()

# Các vectơ hỗ trợ nằm đúng trên hai mép lề, nơi f(x) bằng 1 hoặc -1
print("So vecto ho tro:", len(svm.support_))
for i in svm.support_:
    f = svm.decision_function(X[i:i+1])[0]
    print(f"diem {i:2d}: x1={X[i,0]:.2f}, x2={X[i,1]:.2f}, "
          f"lop {y[i]}, f(x)={f:+.3f}")
print()

# Những điểm khác nằm ngoài lề, nên |f(x)| lớn hơn 1
f_tat_ca = svm.decision_function(X)
ngoai = np.setdiff1d(np.arange(len(X)), svm.support_)
nho_nhat = np.abs(f_tat_ca[ngoai]).min()
print(f"|f(x)| nho nhat cua cac diem khac: {nho_nhat:.3f}")
print()

# Học lại chỉ với các vectơ hỗ trợ
chi_sv = SVC(kernel="linear", C=1000)
chi_sv.fit(X[svm.support_], y[svm.support_])
w2 = chi_sv.coef_[0]
b2 = chi_sv.intercept_[0]
print("Hoc lai chi voi vecto ho tro:")
print(f"w1 = {w2[0]:.3f}, w2 = {w2[1]:.3f}, b = {b2:.3f}")
print()

# Dự đoán cho hai điểm mới
moi = np.array([[3.0, 3.0], [5.0, 4.5]])
for diem, f, lop in zip(moi, svm.decision_function(moi), svm.predict(moi)):
    print(f"Diem ({diem[0]}, {diem[1]}): f(x)={f:+.3f}, doan lop {lop}")

w1 = -0.831, w2 = -0.355, b = 4.820
Do rong cua le = 2 / 0.9039 = 2.2126

So vecto ho tro: 3
diem 18: x1=5.20, x2=4.22, lop 0, f(x)=-1.000
diem 10: x1=2.88, x2=4.02, lop 1, f(x)=+1.000
diem 13: x1=3.59, x2=2.36, lop 1, f(x)=+1.000

|f(x)| nho nhat cua cac diem khac: 1.059

Hoc lai chi voi vecto ho tro:
w1 = -0.831, w2 = -0.355, b = 4.821

Diem (3.0, 3.0): f(x)=+1.262, doan lop 1
Diem (5.0, 4.5): f(x)=-0.933, doan lop 0


In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv("sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("Tap hoc co", len(X_tr), "ban")
print(f"{'C':>6} {'so vecto HT':>12} {'do rong le':>12} {'tap hoc':>10} {'tap kiem tra':>12}")

for C in [0.01, 0.1, 1, 10, 100]:
    mh = make_pipeline(StandardScaler(), SVC(kernel="linear", C=C))
    mh.fit(X_tr, y_tr)
    svm = mh[-1]  # bước cuối của pipeline là SVM
    le = 2 / np.linalg.norm(svm.coef_[0])
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"{C:6} {len(svm.support_):12d} {le:12.2f} {a_hoc:10.4f} {a_kt:12.4f}")

print()

# Bộ sinh viên mở rộng có thêm cột chỉ tiêu, tính bằng nghìn đồng
df3 = pd.read_csv("sinh_vien_mo_rong.csv")
X3 = df3[["gio_on", "diem_giua_ky", "chi_tieu"]]
y3 = df3["qua_mon"]
X3_tr, X3_te, y3_tr, y3_te = train_test_split(
    X3, y3, test_size=0.25, random_state=17, stratify=y3
)

tho = SVC().fit(X3_tr, y3_tr)
chuan = make_pipeline(StandardScaler(), SVC()).fit(X3_tr, y3_tr)

print("Ba cot, kernel RBF mac dinh")
print(f"Doan bua ca lop qua : {y3_te.mean():.4f}")
print(f"Khong chuan hoa     : {accuracy_score(y3_te, tho.predict(X3_te)):.4f}")
print(f"Co chuan hoa        : {accuracy_score(y3_te, chuan.predict(X3_te)):.4f}")

Tap hoc co 90 ban
     C  so vecto HT   do rong le    tap hoc tap kiem tra
  0.01           74         3.84     0.8556       0.8667
   0.1           44         1.70     0.8889       0.9000
     1           31         1.29     0.8778       0.9000
    10           28         1.06     0.9000       0.9000
   100           27         1.05     0.9000       0.9000

Ba cot, kernel RBF mac dinh
Doan bua ca lop qua : 0.6000
Khong chuan hoa     : 0.6000
Co chuan hoa        : 0.9000


In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Phần 1: dữ liệu một chiều, lớp 1 nằm giữa, lớp 0 ở hai đầu
x = np.array([-4.0, -3.4, -2.8, -2.2, -1.3, -0.7, -0.2,
              0.3, 0.8, 1.3, 2.3, 2.9, 3.5, 4.0])
y = (np.abs(x) < 1.8).astype(int)

# Thử mọi ngưỡng: đoán lớp 1 khi x nhỏ hơn hoặc bằng ngưỡng
tot = max(((x <= t).astype(int) == y).mean() for t in x)
print(f"Mot ngưỡng tren truc x, tot nhat dung {tot:.4f}")

# Thêm cột x bình phương rồi dùng SVM tuyến tính
X_moi = np.c_[x, x ** 2]
svm = SVC(kernel="linear", C=1000).fit(X_moi, y)
print(f"Them cot x^2, SVM tuyen tinh dung {svm.score(X_moi, y):.4f}")
print()

# Phần 2: hai vầng trăng, để kernel tự nâng chiều
df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

mo_hinh = {
    "SVM kernel tuyen tinh": SVC(kernel="linear"),
    "SVM kernel da thuc bac 3": SVC(kernel="poly", degree=3),
    "SVM kernel RBF": SVC(kernel="rbf"),
    "Hoi quy logistic": LogisticRegression(),
    "KNN, k = 13": KNeighborsClassifier(n_neighbors=13),
}

print("Hai vang trang, accuracy tren tap kiem tra")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f"  {ten:26s} {acc:.4f}")
print()

rbf = mo_hinh["SVM kernel RBF"]
print("So vecto ho tro cua kernel RBF:", len(rbf.support_), "tren", len(X_tr), "diem hoc")

Mot ngưỡng tren truc x, tot nhat dung 0.7143
Them cot x^2, SVM tuyen tinh dung 1.0000

Hai vang trang, accuracy tren tap kiem tra
  SVM kernel tuyen tinh      0.8667
  SVM kernel da thuc bac 3   0.8667
  SVM kernel RBF             0.9200
  Hoi quy logistic           0.8533
  KNN, k = 13                0.9200

So vecto ho tro cua kernel RBF: 89 tren 225 diem hoc


In [8]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print(f"{'gamma':>6} {'tap hoc':>10} {'tap kiem tra':>14} {'so vecto HT':>14}")
for g in [0.1, 1, 10, 100]:
    mh = SVC(kernel="rbf", gamma=g).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"{g:6.1f} {a_hoc:10.4f} {a_kt:14.4f} {len(mh.support_):14d}")
print()

# Dò lưới: thử mọi cặp bằng kiểm định chéo 5 phần, chỉ trên tập học
luoi = {"C": [0.1, 1, 10, 100], "gamma": [0.01, 0.1, 1, 10]}
tim = GridSearchCV(SVC(kernel="rbf"), luoi, cv=5)
tim.fit(X_tr, y_tr)

bang = pd.DataFrame(tim.cv_results_["params"])
bang["diem"] = tim.cv_results_["mean_test_score"]
print("Diem kiem dinh cheo, hang la C, cot la gamma")
print(bang.pivot(index="C", columns="gamma", values="diem").round(4).to_string())
print()

print("Cap tot nhat:", tim.best_params_)
print(f"Diem kiem dinh cheo: {tim.best_score_:.4f}")
# GridSearchCV đã tự học lại với cặp tốt nhất trên toàn tập học
print(f"Accuracy tren tap kiem tra: {tim.score(X_te, y_te):.4f}")

 gamma    tap hoc   tap kiem tra    so vecto HT
   0.1     0.8356         0.8533            113
   1.0     0.8711         0.9200             88
  10.0     0.8978         0.9067            128
 100.0     0.9422         0.8133            217

Diem kiem dinh cheo, hang la C, cot la gamma
gamma   0.01    0.10    1.00    10.00
C                                    
0.1    0.6578  0.8044  0.8444  0.8578
1.0    0.8044  0.8178  0.8667  0.8667
10.0   0.8178  0.8400  0.8711  0.8311
100.0  0.8178  0.8533  0.8711  0.7467

Cap tot nhat: {'C': 10, 'gamma': 1}
Diem kiem dinh cheo: 0.8711
Accuracy tren tap kiem tra: 0.9200


In [9]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
print("Kich thuoc X:", X.shape)
print("Do sang nho nhat va lon nhat:", X.min(), X.max())
print("So anh moi chu so:", np.bincount(y))
print()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

svm = SVC(kernel="rbf").fit(X_tr, y_tr)
du_doan = svm.predict(X_te)
print(f"SVM kernel RBF: accuracy {accuracy_score(y_te, du_doan):.4f}")
print("So vecto ho tro:", len(svm.support_), "tren", len(X_tr), "anh")
sai = np.flatnonzero(du_doan != y_te)
print("So anh doan sai:", len(sai), "tren", len(y_te))
for i in sai:
    print(f"  anh {i:3d}: that la {y_te[i]}, doan la {du_doan[i]}")
print()

# So với các mô hình khác
mo_hinh = {
    "SVM co chuan hoa": make_pipeline(StandardScaler(), SVC()),
    "KNN, k = 5": KNeighborsClassifier(),
    "Logistic co chuan hoa": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
}

print("Cac mo hinh khac, accuracy tren tap kiem tra")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f"  {ten:22s} {acc:.4f}")

Kich thuoc X: (1797, 64)
Do sang nho nhat va lon nhat: 0.0 16.0
So anh moi chu so: [178 182 177 183 181 182 181 179 174 180]

SVM kernel RBF: accuracy 0.9911
So vecto ho tro: 626 tren 1347 anh
So anh doan sai: 4 tren 450
  anh   6: that la 9, doan la 3
  anh  18: that la 8, doan la 1
  anh 161: that la 4, doan la 8
  anh 428: that la 7, doan la 9

Cac mo hinh khac, accuracy tren tap kiem tra
  SVM co chuan hoa       0.9778
  KNN, k = 5             0.9867
  Logistic co chuan hoa  0.9711


In [13]:
# Bài 1
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]], cluster_std=0.9, random_state=0)
y = 1 - y
svm = SVC(kernel="linear", C=1000)
svm.fit(X, y)

w = svm.coef_[0]
b = svm.intercept_[0]
w1, w2 = w[0], w[1]

moi = np.array([[3.0, 3.0], [5.0, 4.5]])
for diem in moi:
    f_tay = w1 * diem[0] + w2 * diem[1] + b
    khoang_cach = abs(f_tay) / np.sqrt(w1**2 + w2**2)
    print(f"Điểm ({diem[0]}, {diem[1]}): f(x) tính tay = {f_tay:.3f}, Khoảng cách = {khoang_cach:.4f}")

print("\nKhoảng cách của ba vectơ hỗ trợ:")
do_dai_w = np.sqrt(w1**2 + w2**2)
do_rong_le = 2 / do_dai_w
for i in svm.support_:
    f_ht = svm.decision_function(X[i:i+1])[0]
    kc_ht = abs(f_ht) / do_dai_w
    print(f"Điểm {i}: |f(x)| = {abs(f_ht):.3f}, Khoảng cách = {kc_ht:.4f} (bằng đúng một nửa độ rộng lề {do_rong_le/2:.4f})")

print("\nGiải thích: Khoảng cách của các vectơ hỗ trợ đến ranh giới luôn bằng đúng một nửa độ rộng lề vì vectơ hỗ trợ nằm ngay trên hai mép lề, mà mỗi mép lề có phương trình f(x) = 1 hoặc f(x) = -1, trong khi độ rộng lề tổng cộng được tính là 2 / ||w||.")

Điểm (3.0, 3.0): f(x) tính tay = 1.262, Khoảng cách = 1.3958
Điểm (5.0, 4.5): f(x) tính tay = -0.933, Khoảng cách = 1.0325

Khoảng cách của ba vectơ hỗ trợ:
Điểm 18: |f(x)| = 1.000, Khoảng cách = 1.1064 (bằng đúng một nửa độ rộng lề 1.1063)
Điểm 10: |f(x)| = 1.000, Khoảng cách = 1.1062 (bằng đúng một nửa độ rộng lề 1.1063)
Điểm 13: |f(x)| = 1.000, Khoảng cách = 1.1065 (bằng đúng một nửa độ rộng lề 1.1063)

Giải thích: Khoảng cách của các vectơ hỗ trợ đến ranh giới luôn bằng đúng một nửa độ rộng lề vì vectơ hỗ trợ nằm ngay trên hai mép lề, mà mỗi mép lề có phương trình f(x) = 1 hoặc f(x) = -1, trong khi độ rộng lề tổng cộng được tính là 2 / ||w||.


In [15]:
# Bài 2
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

X_goc, y_goc = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]], cluster_std=0.9, random_state=0)
y_goc = 1 - y_goc

X1 = np.vstack([X_goc, [-1, -1]])
y1 = np.append(y_goc, 1)
svm1 = SVC(kernel="linear", C=1000).fit(X1, y1)
w1_kq, b1_kq = svm1.coef_[0], svm1.intercept_[0]

X2 = np.vstack([X_goc, [4.5, 4]])
y2 = np.append(y_goc, 1)
svm2 = SVC(kernel="linear", C=1000).fit(X2, y2)
w2_kq, b2_kq = svm2.coef_[0], svm2.intercept_[0]

print(f"Thêm (-1, -1) -> w1={w1_kq[0]:.3f}, w2={w1_kq[1]:.3f}, b={b1_kq:.3f}")
print(f"Thêm (4.5, 4) -> w1={w2_kq[0]:.3f}, w2={w2_kq[1]:.3f}, b={b2_kq:.3f}")
print("\nNhận xét: Khi thêm điểm (-1, -1) nằm rất sâu về phía lớp 1 và ở xa biên, điểm này không nằm trên mép lề nên không phải là vectơ hỗ trợ, do đó đường ranh giới KHÔNG thay đổi so với ban đầu. Ngược lại, khi thêm điểm (4.5, 4) nằm ngay vùng trung gian giữa hai lớp, nó trở thành vectơ hỗ trợ chi phối và làm ĐƯỜNG RANH GIỚI THAY ĐỔI rõ rệt.")

Thêm (-1, -1) -> w1=-0.831, w2=-0.355, b=4.820
Thêm (4.5, 4) -> w1=-2.603, w2=-0.807, b=15.941

Nhận xét: Khi thêm điểm (-1, -1) nằm rất sâu về phía lớp 1 và ở xa biên, điểm này không nằm trên mép lề nên không phải là vectơ hỗ trợ, do đó đường ranh giới KHÔNG thay đổi so với ban đầu. Ngược lại, khi thêm điểm (4.5, 4) nằm ngay vùng trung gian giữa hai lớp, nó trở thành vectơ hỗ trợ chi phối và làm ĐƯỜNG RANH GIỚI THAY ĐỔI rõ rệt.


In [16]:
# Bài 3
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=17, stratify=y)

for d in [2, 3, 4]:
    pipeline = make_pipeline(StandardScaler(), SVC(kernel="poly", degree=d))
    luoi = {"svc__C": [0.1, 1, 10, 100]}
    tim = GridSearchCV(pipeline, luoi, cv=5)
    tim.fit(X_tr, y_tr)
    acc = tim.score(X_te, y_te)
    print(f"Degree d = {d}: Best C = {tim.best_params_['svc__C']}, Accuracy tập kiểm tra = {acc:.4f}")

print("\nNhận xét: Bậc 3 và bậc 4 cho kết quả tốt, nhưng không vượt qua được kernel RBF ở mục 5 (accuracy RBF đạt 0.9200) vì hình dạng dữ liệu hai vầng trăng phù hợp nhất với bản chất phân phối bởi hàm mũ của RBF.")

Degree d = 2: Best C = 1, Accuracy tập kiểm tra = 0.5067
Degree d = 3: Best C = 1, Accuracy tập kiểm tra = 0.8267
Degree d = 4: Best C = 100, Accuracy tập kiểm tra = 0.4800

Nhận xét: Bậc 3 và bậc 4 cho kết quả tốt, nhưng không vượt qua được kernel RBF ở mục 5 (accuracy RBF đạt 0.9200) vì hình dạng dữ liệu hai vầng trăng phù hợp nhất với bản chất phân phối bởi hàm mũ của RBF.


In [17]:
# Bài 4
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=17, stratify=y)

luoi_min = {"C": [3, 10, 30], "gamma": [0.3, 1, 3]}
tim_min = GridSearchCV(SVC(kernel="rbf"), luoi_min, cv=5)
tim_min.fit(X_tr, y_tr)

print(f"Cặp tham số tốt nhất sau khi dò mịn: {tim_min.best_params_}")
print(f"Điểm kiểm định chéo tốt nhất: {tim_min.best_score_:.4f}")
print(f"Accuracy trên tập kiểm tra: {tim_min.score(X_te, y_te):.4f}")
print("\nNhận xét: Điểm kiểm định chéo cao hơn một chút so với mức cũ (0.8711), tuy nhiên accuracy trên tập kiểm tra vẫn giữ nguyên ở mức 0.9200. Điều này cho thấy dò lưới quá mịn ở gần vùng tối ưu chỉ cải thiện không đáng kể do độ chính xác của mô hình trên tập kiểm tra đã đạt ngưỡng giới hạn của dữ liệu.")

Cặp tham số tốt nhất sau khi dò mịn: {'C': 30, 'gamma': 1}
Điểm kiểm định chéo tốt nhất: 0.8756
Accuracy trên tập kiểm tra: 0.9200

Nhận xét: Điểm kiểm định chéo cao hơn một chút so với mức cũ (0.8711), tuy nhiên accuracy trên tập kiểm tra vẫn giữ nguyên ở mức 0.9200. Điều này cho thấy dò lưới quá mịn ở gần vùng tối ưu chỉ cải thiện không đáng kể do độ chính xác của mô hình trên tập kiểm tra đã đạt ngưỡng giới hạn của dữ liệu.


In [18]:
# Bài 5
import numpy as np
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=17, stratify=y)

svm = SVC(kernel="rbf").fit(X_tr, y_tr)
du_doan = svm.predict(X_te)
cm = confusion_matrix(y_te, du_doan)

print("Bảng nhầm lẫn (Confusion Matrix):")
print(cm)

sai = np.flatnonzero(du_doan != y_te)
print(f"\nTổng số ảnh đoán sai: {len(sai)} trên {len(y_te)} ảnh.")
for i in sai[:5]:
    print(f"Chỉ số ảnh: {i} | Thật là: {y_te[i]} | Đoán là: {du_doan[i]}")

print("\nNhận xét: Nhìn vào bảng nhầm lẫn và các ảnh đoán sai, các chữ số hay bị nhầm lẫn với nhau thường là những chữ số có nét vẽ tương tự nhau khi viết tay (ví dụ: số 9 nhầm thành số 3, số 8 nhầm thành số 1, hoặc số 4 nhầm thành số 8) do đặc điểm độ sáng của các điểm ảnh bị chồng chéo.")


Bảng nhầm lẫn (Confusion Matrix):
[[45  0  0  0  0  0  0  0  0  0]
 [ 0 46  0  0  0  0  0  0  0  0]
 [ 0  0 44  0  0  0  0  0  0  0]
 [ 0  0  0 46  0  0  0  0  0  0]
 [ 0  0  0  0 44  0  0  0  1  0]
 [ 0  0  0  0  0 46  0  0  0  0]
 [ 0  0  0  0  0  0 45  0  0  0]
 [ 0  0  0  0  0  0  0 44  0  1]
 [ 0  1  0  0  0  0  0  0 42  0]
 [ 0  0  0  1  0  0  0  0  0 44]]

Tổng số ảnh đoán sai: 4 trên 450 ảnh.
Chỉ số ảnh: 6 | Thật là: 9 | Đoán là: 3
Chỉ số ảnh: 18 | Thật là: 8 | Đoán là: 1
Chỉ số ảnh: 161 | Thật là: 4 | Đoán là: 8
Chỉ số ảnh: 428 | Thật là: 7 | Đoán là: 9

Nhận xét: Nhìn vào bảng nhầm lẫn và các ảnh đoán sai, các chữ số hay bị nhầm lẫn với nhau thường là những chữ số có nét vẽ tương tự nhau khi viết tay (ví dụ: số 9 nhầm thành số 3, số 8 nhầm thành số 1, hoặc số 4 nhầm thành số 8) do đặc điểm độ sáng của các điểm ảnh bị chồng chéo.


In [19]:
# Bài 6
import time
import numpy as np
from sklearn.datasets import make_classification
from sklearn.svm import SVC, LinearSVC

print(f"{'Số lượng (n)':<15} {'Thời gian SVC (s)':<22} {'Thời gian LinearSVC (s)':<25}")

for n in [2000, 5000, 10000, 20000]:
    X, y = make_classification(n_samples=n, n_features=20, random_state=0)

    t0 = time.time()
    SVC().fit(X, y)
    t_svc = time.time() - t0

    t0 = time.time()
    LinearSVC(dual=False, max_iter=2000).fit(X, y)
    t_linear = time.time() - t0

    print(f"{n:<15} {t_svc:<22.4f} {t_linear:<25.4f}")

print("\nNhận xét: Khi số lượng dữ liệu tăng gấp đôi, thời gian học của SVC (kernel RBF) tăng lên rất nhanh (khoảng 3 đến 4 lần hoặc hơn do phải tính toán ma trận kernel bậc hai), trong khi LinearSVC tăng trưởng thời gian chậm hơn nhiều và hiệu quả rõ rệt trên dữ liệu lớn.")

Số lượng (n)    Thời gian SVC (s)      Thời gian LinearSVC (s)  
2000            0.0573                 0.0073                   
5000            0.2940                 0.0084                   
10000           3.6102                 0.0150                   
20000           11.6204                0.0292                   

Nhận xét: Khi số lượng dữ liệu tăng gấp đôi, thời gian học của SVC (kernel RBF) tăng lên rất nhanh (khoảng 3 đến 4 lần hoặc hơn do phải tính toán ma trận kernel bậc hai), trong khi LinearSVC tăng trưởng thời gian chậm hơn nhiều và hiệu quả rõ rệt trên dữ liệu lớn.
